# PA4 Task 3 — Kaggle runner (exact speculative sampling, off/on-policy draft distillation)

One Kaggle version (**Save Version → Save & Run All (Commit)**, ≤ 12 h; expected ≈ 5 h):

| Phase | GPU 0 | GPU 1 |
|---|---|---|
| A | sampler validation (stops the GPU-0 chain if it fails) → base prefix banks → target + original quality | teacher pools + 4 distillation runs (off/on-policy × seeds 6304/6305) → merge adapters |
| B | student rollout banks → alignment on all banks | trained-draft quality |
| C (timing only) | benchmarks: original (+ isolated costs), off-policy s6304, on-policy s6305 | benchmarks: off-policy s6305, on-policy s6304 → unmerged-adapter timing |

All speculative benchmarks run in phase C, with one benchmark on each GPU at the same time, so every draft is timed
under the same background load (the speedups are additionally paired with an adjacent target-only run).

Then the report and a zip. Every step skips finished work: to resume, add this notebook's output as input, set
`PREV_OUTPUT`, and run again. Settings: **GPU T4 ×2**, **Internet On**.

In [ ]:
# ---------------- settings ----------------
REPO_URL    = "https://github.com/wasaymj/ATML.git"
BRANCH      = "main"
PREV_OUTPUT = ""          # e.g. "/kaggle/input/pa4-task3-runner" to resume

import os, subprocess, sys, time, shutil, glob, json
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA4/Task 3"
DATA = f"{REPO}/PA4/atml-pa4-datasets"
LOGS = f"{WORK}/logs"
ART  = f"{WORK}/t3_artifacts"            # adapters, merged checkpoints, teacher pools (not in git)
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(shutil.disk_usage(WORK))

In [ ]:
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)
print(subprocess.run([sys.executable, "-c", "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ---------------- resume: restore previous outputs ----------------
def copy_tree(src, dst):
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True); print("restored", src)
if PREV_OUTPUT:
    copy_tree(f"{PREV_OUTPUT}/ATML/PA4/Task 3/results", f"{TASK}/results")
    copy_tree(f"{PREV_OUTPUT}/t3_artifacts", ART)
    copy_tree(f"{PREV_OUTPUT}/ATML/PA4/atml-pa4-datasets/data/public_locked", f"{DATA}/data/public_locked")

## Locked revisions, models and data
Both models come from the revision lock committed after Task 1 (`model_target`, `model_draft`). GSM8K rows are
selected exactly as `setup_data.py` does (supplied manifests, locked dataset revision); C4 is not needed.
The staff `--prefix-banks` command is **not** used: on transformers 4.57.1 it fails (`generate(generator=…)`) and it
would sample with the checkpoints' repetition penalty 1.1; our code regenerates the banks with the same protocol at
true temperature 1 (see `src/banks.py`).

In [ ]:
lock = f"{DATA}/data/local_source_revisions.json"
rev = json.load(open(lock))["revisions"]
from huggingface_hub import snapshot_download
for repo_id, key in [("Qwen/Qwen2.5-1.5B-Instruct", "model_target"), ("Qwen/Qwen2.5-0.5B-Instruct", "model_draft")]:
    print(repo_id, rev[key], snapshot_download(repo_id, revision=rev[key]))
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_PROGRESS_BARS="1")
r = subprocess.run([sys.executable, "scripts/run_task3.py", "--step", "data", "--artifacts", ART], cwd=TASK, env=ENV)
assert r.returncode == 0, "data step failed"
subprocess.run([sys.executable, "scripts/validate_assets.py"], cwd=DATA, check=True)

In [ ]:
COMMON = f'--artifacts "{ART}"'

def launch(name, gpu, steps, stop_on_fail=False):
    script = f"{LOGS}/{name}.sh"
    cmds = [f'{sys.executable} scripts/run_task3.py --step {s} {COMMON}' for s in steps]
    if stop_on_fail:
        body = " && ".join(cmds) + ' || echo "STEP FAILED (chain stopped)"'
    else:
        body = "\n".join(c + f' || echo "STEP FAILED: {c.split("--step ")[1].split(" ")[0]}"' for c in cmds)
    open(script, "w").write(f'cd "{TASK}"\n' + body + "\n")
    print(f"[{name}] GPU {gpu}: {steps}", flush=True)
    return subprocess.Popen(["bash", script], stdout=open(f"{LOGS}/{name}.log", "a"), stderr=subprocess.STDOUT,
                            env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=10):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name in procs:
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-3:]
            print(f"--- {name} ({time.strftime('%H:%M')}):", *tail, sep="\n   ", flush=True)
    print({n: p.returncode for n, p in procs.items()})

## Phase A

In [ ]:
import torch
TRAINED = "off_policy_s6304,off_policy_s6305,on_policy_s6304,on_policy_s6305"
if torch.cuda.device_count() >= 2:
    wait({"gpu0_A": launch("gpu0_A", 0, ["validate", "banks_base", "quality --models target,original"], stop_on_fail=True),
          "gpu1_A": launch("gpu1_A", 1, ["train", "merge"], stop_on_fail=True)})
else:
    wait({"gpu0_A": launch("gpu0_A", 0, ["validate", "banks_base", "train", "merge", "quality --models target,original"], stop_on_fail=True)})
print(open(f"{TASK}/results/validation/validation.json").read()[:300] if os.path.exists(f"{TASK}/results/validation/validation.json") else "no validation file")

## Phase B

In [ ]:
if torch.cuda.device_count() >= 2:
    wait({"gpu0_B": launch("gpu0_B", 0, ["banks_students", "align"]),
          "gpu1_B": launch("gpu1_B", 1, [f"quality --models {TRAINED}"])})
else:
    wait({"gpu0_B": launch("gpu0_B", 0, ["banks_students", "align", f"quality --models {TRAINED}"])})

## Phase C (benchmarks)

In [ ]:
# Phase C: timing only. One benchmark per GPU at the same time (equal background load for every draft).
if torch.cuda.device_count() >= 2:
    wait({"gpu0_C": launch("gpu0_C", 0, ["bench --drafts original,off_policy_s6304,on_policy_s6305"]),
          "gpu1_C": launch("gpu1_C", 1, ["bench --drafts off_policy_s6305,on_policy_s6304", "bench_unmerged"])})
else:
    wait({"gpu0_C": launch("gpu0_C", 0, [f"bench --drafts original,{TRAINED}", "bench_unmerged"])})
for f in glob.glob(f"{LOGS}/*.log"):
    print(f, "<-- contains a failed step" if "STEP FAILED" in open(f, errors="replace").read() else "ok")

In [ ]:
# ---------------- report + packaging (display comes after, in its own cell) ----------------
subprocess.run([sys.executable, "scripts/run_task3.py", "--step", "status", "--artifacts", ART], cwd=TASK)
subprocess.run([sys.executable, "scripts/run_task3.py", "--step", "report"], cwd=TASK)
shutil.copytree(LOGS, f"{TASK}/results/logs", dirs_exist_ok=True)
shutil.make_archive(f"{WORK}/task3_results", "zip", TASK, "results")
if os.path.isdir(f"{ART}/adapters"):
    shutil.make_archive(f"{WORK}/task3_adapters", "zip", ART, "adapters")
print(sorted(os.listdir(WORK)))

In [ ]:
from IPython.display import Markdown, Image, display
for t in sorted(glob.glob(f"{TASK}/results/tables/*.md")):
    display(Markdown(f"### {os.path.basename(t)}" + chr(10) + open(t, encoding="utf-8").read()))
for fig in sorted(glob.glob(f"{TASK}/results/figures/*.png")):
    display(Image(fig, width=900))